In [ ]:
import pandas as pd
df = pd.read_csv('data/budgetwise_synthetic_dirty.csv')
df.shape

(15836, 9)

In [3]:
df.isnull().sum()

transaction_id         0
user_id                0
date                 344
transaction_type       0
category             158
amount               178
payment_mode         503
location             722
notes               1534
dtype: int64

In [4]:
# check for duplicate rows
df.duplicated().sum()

np.int64(804)

In [5]:
# remove duplicate rows
print("Rows before:", len(df))

df = df.drop_duplicates()

print("Rows after:", len(df))

Rows before: 15836
Rows after: 15032


In [6]:
df.head(10)

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes
0,T03512,U039,December 22 2021,Expense,Rent,998,Cash,Pune,Paid electricity bill
1,T03261,U179,03/24/2022,Expense,Food,$143,Card,Delhi,Grocery shopping
2,T04316,U143,October 18 2022,Expense,Rent,149,Cash,Bengaluru,NaN
3,T05649,U079,12/12/2021,Expense,Rent,49,UPI,NaN,Paid electricity bill
4,T14750,U020,NaN,Income,Other Income,"83,802",Bank Transfer,Chennai,Gift via app
5,T04246,U188,12-07-22,Expense,Entertainment,208,UPI,Hyderabad,Monthly rent
6,T01814,U150,September 12 2019,Expense,Savings,681,Cash,Kolkata,Salary
7,T06293,U014,2022-01-06,Expense,Rent,"2,524",Bank Transfer,NaN,Salary
8,T07930,U138,04-11-22,Expense,Education,93,Bank Transfer,Pune,hZtATyn1UX55solCMr1
9,T04656,U112,February 03 2021,Expense,Food,323,UPI,Ahmedabad,Monthly rent


In [7]:
# check for missing values in the 'date' column
df['date'].isnull().sum()

np.int64(327)

In [8]:
# standardize date format
df['date'] = pd.to_datetime(
    df['date'],
    format='mixed', 
    errors='coerce')

In [9]:
# check for unique values in the 'transaction_type' column
df['transaction_type'].unique()

<ArrowStringArray>
['Expense', 'Income']
Length: 2, dtype: str

In [10]:
# check for unique values in the 'category' column
df['category'].unique()

<ArrowStringArray>
[          'Rent',           'Food',   'Other Income',  'Entertainment',
        'Savings',      'Education',         'Others', 'Entertainmennt',
        'Salaryy',         'Travel',
 ...
         'Salray',   'Other Incmoe',    'Oher Income',   'Entertainmet',
          'Healh',    'Other Incoe',        'Savinsg',        'Otherss',
         'Slaary',     'Educatioon']
Length: 213, dtype: str

In [11]:
# check the frequency of each category to create a mapping for similar categories
df['category'].value_counts().head(50)

category
Food             3559
Rent             2922
Travel           1382
Utilities        1226
Entertainment    1014
Bonus             709
Other Income      707
Salary            707
Others            576
Savings           551
Education         328
Health            222
Fod                47
Foood              45
FFood              29
Ret                24
eRnt               23
Fodo               23
Rentt              23
Foo                22
Reent              20
Ren                19
Retn               19
Foodd              19
oFod               19
ent                19
ood                16
Rennt              16
Rnet               16
Rnt                14
RRent              14
Tavel              12
Trave               9
Utilitiees          9
ravel               8
Tarvel              8
Trvael              8
Salaryy             7
rTavel              7
Travle              7
Trvel               7
Travell             7
aSvings             6
Utilitise           6
Bonu                6
S

In [12]:
pip install rapidfuzz


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [13]:
# use rapidfuzz to find similar categories and create a new column with cleaned categories
from rapidfuzz import process

valid_categories = [
    'Food',
    'Rent',
    'Travel',
    'Utilities',
    'Entertainment',
    'Bonus',
    'Salary',
    'Other Income',
    'Others',
    'Savings',
    'Education',
    'Health'
]

def fix_category(cat):
    if pd.isna(cat):
        return cat

    match, score, _ = process.extractOne(
        cat,
        valid_categories
    )

    if score >= 80:
        return match

    return cat

df['category_clean'] = df['category'].apply(fix_category)

In [14]:
df['category_clean'].unique()

<ArrowStringArray>
[         'Rent',          'Food',  'Other Income', 'Entertainment',
       'Savings',     'Education',        'Others',        'Salary',
        'Travel',         'Bonus',        'Health',     'Utilities',
             nan,          'eRnt',          'Retn',          'oFod',
          'Rnet',          'Fodo']
Length: 18, dtype: str

In [15]:
#  Add manually map remaining similar categories
category_mapping = {
    'eRnt': 'Rent',
    'Retn': 'Rent',
    'Rnet': 'Rent',
    'oFod': 'Food',
    'Fodo': 'Food'
}
df['category_clean'] = df['category_clean'].replace(category_mapping)

In [16]:
# finalize the cleaned category column and drop the temporary column
df['category'] = df['category_clean']
df.drop(columns=['category_clean'], inplace=True)

In [17]:
# check for missing values in the 'amount' column
df['amount'].isnull().sum()

np.int64(170)

In [18]:
# Return rows where 'amount' cannot be converted to numeric
bad_rows = df[pd.to_numeric(df['amount'], errors='coerce').isna()]
print(bad_rows['amount'].unique())

<ArrowStringArray>
[  '$143', '83,802',  '2,524',  '1,019', '59,149', '78,744', '58,874',
   '₹383',  '1,761',  '1,580',
 ...
 '49,076', '₹1,971', '74,323',  '2,370', '59,397', '48,170',   '$278',
 '₹1,324', '52,674',   '₹432']
Length: 5256, dtype: str


In [19]:
# clean the 'amount' column by removing currency symbols and commas, then convert to numeric
df['amount'] = (df['amount']
    .astype(str)
    .str.replace('[$,₹]', '', regex=True)
    .str.strip()
)
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')

In [20]:
# check for unique values in the 'payment_mode' column and their counts to identify right values and potential typos
df['payment_mode'].value_counts()

payment_mode
Bank Transfer    3633
Cash             3597
UPI              3539
Card             3536
UI                 14
                 ... 
Bank Transfr        1
Bank Trnasfer       1
Bank Tarnsfer       1
Bank Transefr       1
ank Transfer        1
Name: count, Length: 62, dtype: int64

In [21]:
valid_payment_modes = [
    'Bank Transfer',
    'Cash', 
    'UPI',
    'Card'
]

def fix_category(cat):
    if pd.isna(cat):
        return cat

    match, score, _ = process.extractOne(
        cat,
        valid_payment_modes
    )

    if score >= 80:
        return match

    return cat

df['payment_mode_clean'] = df['payment_mode'].apply(fix_category)

In [22]:
df['payment_mode_clean'].unique()

<ArrowStringArray>
[         'Cash',          'Card',           'UPI', 'Bank Transfer',
             nan,          'aCrd',          'Cadr',           'UIP',
          'Csah',          'Crad',          'aCsh',           'PUI',
          'Cahs']
Length: 13, dtype: str

In [23]:
# Add manually map remaining similar payment modes
mode_mapping = {
    'aCrd': 'Card',
    'Cadr': 'Card',
    'UIP': 'UPI',
    'Csah': 'Cash',
    'Crad': 'Card',
    'aCsh': 'Cash',
    'PUI': 'UPI',
    'Cahs': 'Cash'
}
df['payment_mode_clean'] = df['payment_mode_clean'].replace(mode_mapping)

In [24]:
df['payment_mode'] = df['payment_mode_clean']
df.drop(columns=['payment_mode_clean'], inplace=True)

In [25]:
# check for unique values in the 'location' column to identify potential typos and inconsistencies
df['location'].unique()

<ArrowStringArray>
[     'Pune',     'Delhi', 'Bengaluru',         nan,   'Chennai', 'Hyderabad',
   'Kolkata', 'Ahmedabad',    'jaipur',   'Lucknow',    'Mumbai',    'mumbai',
    'Jaipur', 'bengaluru',     'delhi',      'pune',    'JAIPUR', 'hyderabad',
   'lucknow',     'DELHI', 'AHMEDABAD', 'ahmedabad', 'HYDERABAD',   'CHENNAI',
   'chennai',   'kolkata',      'PUNE', 'BENGALURU',    'MUMBAI',   'KOLKATA',
   'LUCKNOW']
Length: 31, dtype: str

In [26]:
df['location'] = (
    df['location']
        .str.strip()
        .str.title()
)

In [27]:
df['notes'].unique()

<ArrowStringArray>
['Paid electricity bill ',       'Grocery shopping',                      nan,
  'Paid electricity bill',           'Gift via app',           'Monthly rent',
                 'Salary',    'hZtATyn1UX55solCMr1',            'Salary cash',
            'Course fee ',
 ...
    'snpgCR3eNCCnNGfHoKc',              'i6S7sewLX',   'vkWeZBZpo8swHiuL0MKO',
             'QGSZoRuXJ3',        'iHBG6efX8afJNQf',               'vu36JLrh',
        'x7SWy2WPYxcprhu',            'iXgyuylgSkf',               'xopbxsPp',
      'Aw5aDyInfbtc9hQta']
Length: 989, dtype: str

In [28]:
import pandas as pd
import re

def clean_notes(note):
    """
    Clean notes column:
    - Replace NaN with empty string
    - Remove leading/trailing whitespace
    - Remove random-looking generated strings
    - Keep legitimate transaction descriptions
    """

    # Handle missing values
    if pd.isna(note):
        return ""

    note = str(note).strip()

    # Handle empty strings
    if note == "":
        return ""

    # Remove suspicious random strings:
    # Examples:
    # sDUschT0
    # iXgyuylgSkf
    # vu36JLrh
    # hZtATyn1UX55solCMr1
    if (
        len(note) >= 8
        and " " not in note
        and (
            re.search(r"\d", note) or
            (
                re.search(r"[A-Z]", note)
                and re.search(r"[a-z]", note)
            )
        )
    ):
        return ""

    return note


# Create cleaned column
df["notes_cleaned"] = df["notes"].apply(clean_notes)

In [29]:
removed_notes = df[
    (df["notes"].notna())
    & (df["notes"].str.strip() != "")
    & (df["notes_cleaned"] == "")
]

print("Removed notes:")
print(removed_notes["notes"].head(50))

Removed notes:
8       hZtATyn1UX55solCMr1
21          wkAdxBaz8rnYSOE
37             VDSr4SI1JVHg
38          A7Brvt0bSrovLXo
45         NLS1gMaaOza0C0cf
53               zQQvRHrCGH
72           cJ9BkudD4Tf3Ql
81                sboT5a6RW
90               kU9tfLsTrc
100               CoBzaIzlT
114                16wmjUjz
122               zldFyXXwJ
127        LDRxleWusbe7ZvYa
136              1yv2UgqhuT
164         nqj5djeLfknN4JU
167      4qk8NxcvcHAcxzQrBB
171        lPujru8v5kb9vKPv
184                iU4GRmkU
186                RQEpVsCm
203             CTahRXPpt1r
217      zhjquyI3MwYbP4rQr9
232              9mR8zTIgFl
250     iu2HPMFwQ3ZsxQWTV25
266             7LFdoPRzhpf
279      fADSGGeLdIUz9KhNcw
292         D3QuRGl5j4yuEvu
300               uGLMTS2HW
333                TcWJmtPa
334               NxH69Odw2
340          ezebfSjgbIcK3D
342      NUGOxY5uEE6VGxC6Dd
345      oUUq1sbTZ0KCLrWUTx
379            GicUyo3QZlua
386        OkSl27OasapS6rCA
410              hBz81no7L1
413  

In [30]:
print("Original unique notes:")
print(df["notes"].nunique())

print("\nCleaned unique notes:")
print(df["notes_cleaned"].nunique())

Original unique notes:
988

Cleaned unique notes:
41


In [31]:
df['notes_cleaned'].value_counts(dropna=True)


notes_cleaned
                                 2384
Gift                             1036
Dinner at resto                  1003
Doctor visit                      996
Netflix subscription              989
Monthly rent                      984
Grocery shopping                  970
Uber to office                    965
Paid electricity bill             962
Course fee                        956
Salary                            951
Doctor visit via app              118
Salary via app                    112
Paid electricity bill late        108
Uber to office cash               108
Gift late                         104
Gift cash                         103
Grocery shopping via app          103
Gift via app                      102
Monthly rent via app              101
Paid electricity bill cash        100
Grocery shopping late             100
Salary cash                        98
Uber to office via app             98
Monthly rent cash                  97
Grocery shopping cash              9

In [32]:
df["notes"] = df["notes_cleaned"]

df.drop(
    columns=["notes_cleaned"],
    inplace=True
)

In [33]:
# Check for duplicates transaction_id
duplicate_ids = df[df.duplicated(subset=['transaction_id'], keep=False)].copy()
print(duplicate_ids.shape[0])

346


In [34]:
duplicate_ids = duplicate_ids.sort_values(by='transaction_id')
duplicate_ids

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes
5482,T00058,U136,2021-12-06,Expense,Rent,26.0,Cash,Mumbai,Salary
9870,T00058,U034,2020-04-10,Expense,Rent,NaN,Cash,Delhi,
12552,T00170,U133,2022-10-08,Income,Salary,54120.0,Card,Bengaluru,Monthly rent
15677,T00170,U133,2022-10-08,Income,NaN,54120.0,Card,Bengaluru,Monthly rent
10841,T00235,U139,2020-12-10,Expense,Rent,278.0,NaN,Jaipur,
...,...,...,...,...,...,...,...,...,...
5123,T14732,U012,2019-03-15,Expense,Others,172.0,Cash,Pune,Netflix subscription
9301,T14732,U093,2019-09-22,Income,Salary,58949.0,Card,Hyderabad,Course fee
897,T14837,U079,2020-11-22,Expense,Rent,429.0,Cash,Lucknow,Grocery shopping
8482,T14837,U098,2019-08-09,Expense,Entertainment,175.0,UPI,Pune,Salary


In [ ]:
import pandas as pd

df = df.drop_duplicates()

# ensure date is datetime
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# completeness score
df["completeness_score"] = df.notna().sum(axis=1)

df = df.sort_values(
    ["completeness_score", "date"],
    ascending=[False, False]  # highest score first, latest date first
).drop_duplicates("transaction_id", keep="first")


In [36]:
df = df[~(
    (df["transaction_id"] == "T00842") &
    (df["category"] == "Utilities")    
)]

In [37]:
df['transaction_id'].duplicated().sum()

np.int64(0)

In [40]:
df.head(20)

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes
2279,T03166,U045,2022-12-31,Expense,Food,291.0,Card,Lucknow,Doctor visit
2358,T10805,U014,2022-12-31,Expense,Travel,60.0,Cash,Lucknow,Uber to office
2829,T04934,U071,2022-12-31,Expense,Food,110.0,Bank Transfer,Jaipur,Monthly rent
2850,T05633,U169,2022-12-31,Income,Other Income,54173.0,Bank Transfer,Chennai,Monthly rent
4595,T08234,U157,2022-12-31,Expense,Travel,305.0,Cash,Kolkata,Course fee
5152,T09484,U189,2022-12-31,Expense,Food,123.0,UPI,Mumbai,Salary
5709,T09466,U032,2022-12-31,Expense,Entertainment,2215.0,Bank Transfer,Bengaluru,
8804,T03693,U128,2022-12-31,Expense,Rent,106.0,Cash,Ahmedabad,Grocery shopping via app
9696,T12382,U081,2022-12-31,Expense,Travel,1525.0,Card,Pune,Course fee
11216,T14253,U064,2022-12-31,Expense,Rent,2940.0,Cash,Jaipur,Grocery shopping


In [39]:
df.drop(columns=['completeness_score'], inplace=True)

In [ ]:
df.to_csv('data/budgetwise_synthetic_cleaned.csv', index=False)

In [34]:
# load the cleaned data and check for missing values in the 'user_id' column
import pandas as pd
df = pd.read_csv('data/budgetwise_synthetic_cleaned.csv')
# df['user_id'].isnull().sum()
df['user_id'].unique()

<StringArray>
['U045', 'U014', 'U071', 'U169', 'U157', 'U189', 'U032', 'U128', 'U081',
 'U064',
 ...
 'U108', 'U037', 'U146', 'U090', 'U180', 'U054', 'U113', 'U083', 'U060',
 'U019']
Length: 192, dtype: str

In [27]:
df['amount'].isnull().sum()

np.int64(0)

In [28]:
df = df.dropna(subset=['amount'])

In [31]:
df['transaction_number'] = df['transaction_id'].str.extract(r'T(\d{5})').astype(int)

In [36]:
df.drop(columns="transaction_id", inplace= True)

In [37]:
df.head()

,user_id,date,transaction_type,category,amount,payment_mode,location,notes,transaction_number
0,U045,2022-12-31,Expense,Food,291.0,Card,Lucknow,Doctor visit,3166
1,U014,2022-12-31,Expense,Travel,60.0,Cash,Lucknow,Uber to office,10805
2,U071,2022-12-31,Expense,Food,110.0,Bank Transfer,Jaipur,Monthly rent,4934
3,U169,2022-12-31,Income,Other Income,54173.0,Bank Transfer,Chennai,Monthly rent,5633
4,U157,2022-12-31,Expense,Travel,305.0,Cash,Kolkata,Course fee,8234


In [38]:
df.to_csv('data/budgetwise_synthetic_cleaned.csv', index=False)